<a href="https://colab.research.google.com/github/Ashonet/Warframe_Trading_Analysis/blob/main/Warframe_Market_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import time

BASE_URL = "https://api.warframe.market/v1"
HEADERS = {
    "accept": "application/json",
    "platform": "pc",
    "language": "en"
}

# Fetch all items from the market
def get_all_items():
    response = requests.get(f"{BASE_URL}/items", headers=HEADERS)
    data = response.json()
    return data["payload"]["items"]

# Get item details to find components (for sets)
def get_item_parts(item_url_name):
    response = requests.get(f"{BASE_URL}/items/{item_url_name}", headers=HEADERS)
    data = response.json()
    item = data["payload"]["item"]
    if not item.get("items_in_set"):
        return []

    parts = []
    for part in item["items_in_set"]:
        if not part["url_name"].endswith("_set"):  # skip self
            parts.append(part["url_name"])
    return parts

def is_set(item_url_name):
    return item_url_name.endswith("_set")

def build_sets_and_parts():
    items = get_all_items()
    sets = [item["url_name"] for item in items if is_set(item["url_name"])]
    sets_dict = {}

    print(f"🔎 Found {len(sets)} sets. Processing...")

    for idx, set_name in enumerate(sets):
        try:
            parts = get_item_parts(set_name)
            if parts:
                sets_dict[set_name] = parts
                print(f"[{idx+1}/{len(sets)}] ✅ {set_name} has {len(parts)} parts.")
            time.sleep(0.25)  # Rate limit
        except Exception as e:
            print(f"❌ Failed to fetch parts for {set_name}: {e}")
            continue

    return sets_dict

def save_to_txt(sets_dict, filename="sets_and_parts.txt"):
    with open(filename, "w") as f:
        for set_name, parts in sets_dict.items():
            f.write(f"{set_name}:\n")
            for part in parts:
                f.write(f"  - {part}\n")
            f.write("\n")
    print(f"\n📄 Saved set-part data to {filename}")

if __name__ == "__main__":
    sets_and_parts = build_sets_and_parts()
    save_to_txt(sets_and_parts)


🔎 Found 226 sets. Processing...
[1/226] ✅ frost_prime_set has 4 parts.
[2/226] ✅ mantis_set has 3 parts.
[3/226] ✅ hydroid_prime_set has 4 parts.
[4/226] ✅ wolf_sledge_set has 4 parts.
[5/226] ✅ vasto_prime_set has 3 parts.
[6/226] ✅ zhuge_prime_set has 5 parts.
[7/226] ✅ dual_kamas_prime_set has 3 parts.
[8/226] ✅ snipetron_vandal_set has 4 parts.
[9/226] ✅ destreza_prime_set has 3 parts.
[10/226] ✅ saryn_prime_set has 4 parts.
[11/226] ✅ stradavar_prime_set has 4 parts.
[12/226] ✅ helios_prime_set has 4 parts.
[13/226] ✅ silva_and_aegis_prime_set has 4 parts.
[14/226] ✅ kavasa_prime_kubrow_collar_set has 3 parts.
[15/226] ✅ latron_prime_set has 4 parts.
[16/226] ✅ lex_prime_set has 3 parts.
[17/226] ✅ odonata_prime_set has 4 parts.
[18/226] ✅ vectis_prime_set has 4 parts.
[19/226] ✅ rhino_prime_set has 4 parts.
[20/226] ✅ sicarus_prime_set has 3 parts.
[21/226] ✅ scimitar_set has 3 parts.
[22/226] ✅ braton_prime_set has 4 parts.
[23/226] ✅ aklex_prime_set has 2 parts.
[24/226] ✅ nink

In [10]:
import requests
import time

BASE_URL = "https://api.warframe.market/v1"
HEADERS = {
    "accept": "application/json",
    "platform": "pc",
    "language": "en"
}

def get_all_items():
    res = requests.get(f"{BASE_URL}/items", headers=HEADERS)
    return res.json()["payload"]["items"]

def get_orders_by_rank(url_name, rank):
    res = requests.get(f"{BASE_URL}/items/{url_name}/orders", headers=HEADERS)
    orders = res.json()["payload"]["orders"]
    return [
        o for o in orders
        if o["order_type"] == "sell" and
           o["user"]["status"] == "ingame" and
           o.get("mod_rank") == rank
    ]

def get_arcanes_with_ranks():
    items = get_all_items()
    arcane_items = [
        i for i in items if "arcane" in i["url_name"] and "helmet" not in i["url_name"]
    ]
    results = {}

    print(f"🔮 Checking {len(arcane_items)} arcanes (excluding helmets)...")

    for idx, arcane in enumerate(arcane_items):
        name = arcane["url_name"]
        try:
            rank0 = get_orders_by_rank(name, 0)
            rank5 = get_orders_by_rank(name, 5)

            if rank0 or rank5:
                results[name] = []
                if rank0:
                    results[name].append("Rank 0/5")
                if rank5:
                    results[name].append("Rank 5/5")
                print(f"[{idx+1}] ✅ {name}")
            time.sleep(0.25)
        except Exception as e:
            print(f"[{idx+1}] ❌ Error for {name}: {e}")
    return results

def save_to_txt(data, filename="arcanes_by_rank.txt"):
    with open(filename, "w") as f:
        for name, ranks in sorted(data.items()):
            f.write(f"{name}:\n")
            for r in ranks:
                f.write(f"  - {r}\n")
            f.write("\n")
    print(f"\n📄 Saved to {filename}")

if __name__ == "__main__":
    arcanes = get_arcanes_with_ranks()
    save_to_txt(arcanes)


🔮 Checking 52 arcanes (excluding helmets)...
[1] ✅ arcane_barrier
[2] ✅ arcane_ice
[3] ✅ arcane_trickery
[4] ✅ arcane_velocity
[5] ✅ arcane_resistance
[6] ✅ arcane_pulse
[7] ✅ arcane_fury
[8] ✅ arcane_aegis
[9] ✅ arcane_consequence
[10] ✅ arcane_energize
[11] ✅ arcane_strike
[12] ✅ arcane_nullifier
[13] ✅ arcane_victory
[14] ✅ arcane_eruption
[15] ✅ arcane_awakening
[16] ✅ arcane_ultimatum
[17] ✅ arcane_deflection
[18] ✅ arcane_grace
[19] ✅ arcane_tempo
[20] ✅ arcane_agility
[21] ✅ arcane_bodyguard
[22] ✅ arcane_primary_charger
[23] ✅ arcane_blade_charger
[24] ✅ arcane_guardian
[25] ✅ arcane_healing
[26] ✅ arcane_warmth
[27] ✅ arcane_acceleration
[28] ✅ arcane_arachne
[29] ✅ arcane_momentum
[30] ✅ arcane_precision
[31] ✅ arcane_tanker
[32] ✅ arcane_pistoleer
[33] ✅ arcane_avenger
[34] ✅ arcane_phantasm
[35] ✅ arcane_rage
[36] ✅ arcane_rise
[37] ✅ arcane_blessing
[38] ✅ arcane_steadfast
[39] ✅ arcane_double_back
[40] ✅ arcane_intention
[41] ✅ arcane_reaper
[42] ✅ arcane_power_ramp
[43] 

In [9]:
import requests
import time

BASE_URL = "https://api.warframe.market/v1"
HEADERS = {
    "accept": "application/json",
    "platform": "pc",
    "language": "en"
}

def load_mods_from_file(filename="mods_by_type.txt"):
    mods = []
    try:
        with open(filename, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("=="):
                    continue
                mods.append(line)
        print(f"📂 Loaded {len(mods)} mods from {filename}")
        return mods
    except FileNotFoundError:
        print(f"❌ File {filename} not found")
        return []

def get_orders_by_rank(url_name, rank):
    try:
        res = requests.get(f"{BASE_URL}/items/{url_name}/orders", headers=HEADERS)
        res.raise_for_status()
        orders = res.json()["payload"]["orders"]
        return [
            o for o in orders
            if o["order_type"] == "sell" and
               o["user"]["status"] == "ingame" and
               o.get("mod_rank") == rank
        ]
    except Exception as e:
        print(f"❌ Error fetching orders for {url_name} rank {rank}: {e}")
        return []

def get_mods_with_ranks():
    mods = load_mods_from_file()
    if not mods:
        return {}

    results = {}
    print(f"🔧 Checking {len(mods)} mods...")

    for idx, mod_name in enumerate(mods):
        url_name = mod_name.lower().replace(" ", "_").replace("'", "")
        try:
            ranks_found = []
            # Check Rank 0
            orders_r0 = get_orders_by_rank(url_name, 0)
            if orders_r0:
                ranks_found.append(0)
                print(f"[{idx+1}] ✅ {mod_name} (Rank 0: {len(orders_r0)} orders)")

            # Check ranks in descending order (10 to 1) to find max rank
            max_rank = None
            for rank in range(10, 0, -1):
                orders = get_orders_by_rank(url_name, rank)
                if orders:
                    EDULE: max_rank = rank
                    ranks_found.append(rank)
                    print(f"[{idx+1}] ✅ {mod_name} (Rank {rank}: {len(orders)} orders)")
                    break  # Stop once max rank is found
                time.sleep(0.5)  # Avoid rate limits

            if ranks_found:
                results[mod_name] = ranks_found
            else:
                print(f"[{idx+1}] ⚠️ {mod_name}: No sell orders found for any rank")
            time.sleep(0.5)
        except Exception as e:
            print(f"[{idx+1}] ❌ Error for {mod_name}: {e}")
    return results

def save_to_txt(data, filename="mods_by_rank.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for mod_name, ranks in sorted(data.items()):
            for rank in ranks:
                f.write(f"{mod_name} rank {rank}\n")
    print(f"\n📄 Saved to {filename}")

if __name__ == "__main__":
    mods = get_mods_with_ranks()
    save_to_txt(mods)

Streaming output truncated to the last 5000 lines.
[245] ✅ Parry (Rank 0: 5 orders)
[245] ✅ Parry (Rank 5: 3 orders)
[246] ✅ Parry (Rank 0: 5 orders)
[246] ✅ Parry (Rank 5: 3 orders)
[247] ✅ Parry (Rank 0: 5 orders)
[247] ✅ Parry (Rank 5: 3 orders)
[248] ✅ Partitioned Mallet (Rank 0: 6 orders)
[248] ✅ Partitioned Mallet (Rank 3: 4 orders)
[249] ✅ Patagium (Rank 0: 9 orders)
[249] ✅ Patagium (Rank 5: 6 orders)
[250] ✅ Path Of Statues (Rank 0: 10 orders)
[250] ✅ Path Of Statues (Rank 3: 6 orders)
[251] ✅ Peaceful Provocation (Rank 0: 2 orders)
[251] ✅ Peaceful Provocation (Rank 3: 6 orders)
[252] ✅ Phoenix Renewal (Rank 0: 7 orders)
[252] ✅ Phoenix Renewal (Rank 3: 7 orders)
[253] ✅ Photon Repeater (Rank 0: 3 orders)
[253] ✅ Photon Repeater (Rank 3: 4 orders)
[254] ✅ Physique (Rank 0: 15 orders)
[254] ✅ Physique (Rank 5: 8 orders)
[255] ✅ Piercing Navigator (Rank 0: 4 orders)
[255] ✅ Piercing Navigator (Rank 3: 8 orders)
[256] ✅ Piercing Roar (Rank 0: 12 orders)
[256] ✅ Piercing Roar (Ra

In [ ]:
import requests
import time

BASE_URL = "https://api.warframe.market/v1"
HEADERS = {
    "accept": "application/json",
    "platform": "pc",
    "language": "en"
}

def load_sets_parts(filename="sets_and_parts.txt"):
    sets = {}
    current_set = None
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if not line.startswith("-"):
                current_set = line.strip(":")
                sets[current_set] = []
            else:
                sets[current_set].append(line.strip("- ").strip())
    return sets

def get_orders(item_url_name):
    url = f"{BASE_URL}/items/{item_url_name}/orders"
    response = requests.get(url, headers=HEADERS)
    orders = response.json()["payload"]["orders"]
    filtered = [o for o in orders if o["order_type"] == "sell" and o["user"]["status"] == "ingame"]
    return sorted(filtered, key=lambda o: o["platinum"])

def get_lowest_sell(item_url_name):
    try:
        orders = get_orders(item_url_name)
        return orders[0]["platinum"] if orders else None
    except Exception as e:
        print(f"❌ Error fetching {item_url_name}: {e}")
        return None

def analyze_sets(sets):
    results = []

    print("\n📦 PRIME SET SPREAD ANALYSIS")
    print("-" * 60)
    for set_name, parts in sets.items():
        set_price = get_lowest_sell(set_name)
        part_prices = [get_lowest_sell(part) for part in parts]
        part_prices = [p for p in part_prices if p is not None]

        if not set_price or not part_prices:
            print(f"{set_name}: Missing price data, skipping.")
            continue

        total_parts = sum(part_prices)
        difference = set_price - total_parts
        percentage = (difference / total_parts) * 100

        print(f"🧩 {set_name}")
        print(f"  Set price:     {set_price}p")
        print(f"  Parts total:   {total_parts:.1f}p")
        print(f"  Difference:    {difference:.1f}p")
        print(f"  % Difference:  {percentage:.1f}%\n")

        results.append({
            "set": set_name,
            "set_price": set_price,
            "parts_price": total_parts,
            "difference": difference,
            "percent_diff": percentage
        })

        time.sleep(0.5)  # Avoid rate-limiting

    return results

def save_results(results, filename="set_spread_analysis.txt"):
    with open(filename, "w") as f:
        f.write("📊 Sorted by Absolute Platinum Difference:\n")
        for r in sorted(results, key=lambda x: x["difference"], reverse=True):
            f.write(
                f"{r['set']}: Set {r['set_price']}p | Parts {r['parts_price']}p | Diff {r['difference']}p | % {r['percent_diff']:.1f}%\n"
            )

        f.write("\n📊 Sorted by Percentage Difference:\n")
        for r in sorted(results, key=lambda x: x["percent_diff"], reverse=True):
            f.write(
                f"{r['set']}: Set {r['set_price']}p | Parts {r['parts_price']}p | Diff {r['difference']}p | % {r['percent_diff']:.1f}%\n"
            )

    print(f"\n📄 Saved to {filename}")

if __name__ == "__main__":
    sets = load_sets_parts("sets_and_parts.txt")
    result_data = analyze_sets(sets)
    save_results(result_data)



📦 PRIME SET SPREAD ANALYSIS
------------------------------------------------------------
🧩 frost_prime_set
  Set price:     90p
  Parts total:   83.0p
  Difference:    7.0p
  % Difference:  8.4%

🧩 mantis_set
  Set price:     1p
  Parts total:   16.0p
  Difference:    -15.0p
  % Difference:  -93.8%

🧩 hydroid_prime_set
  Set price:     95p
  Parts total:   81.0p
  Difference:    14.0p
  % Difference:  17.3%

🧩 wolf_sledge_set
  Set price:     17p
  Parts total:   19.0p
  Difference:    -2.0p
  % Difference:  -10.5%

🧩 vasto_prime_set
  Set price:     30p
  Parts total:   25.0p
  Difference:    5.0p
  % Difference:  20.0%

🧩 zhuge_prime_set
  Set price:     55p
  Parts total:   89.0p
  Difference:    -34.0p
  % Difference:  -38.2%

🧩 dual_kamas_prime_set
  Set price:     100p
  Parts total:   47.0p
  Difference:    53.0p
  % Difference:  112.8%

🧩 snipetron_vandal_set
  Set price:     40p
  Parts total:   44.0p
  Difference:    -4.0p
  % Difference:  -9.1%

🧩 destreza_prime_set
  Set p

In [ ]:
import requests
import time
from collections import defaultdict

BASE_URL = "https://api.warframe.market/v1"
HEADERS = {
    "accept": "application/json",
    "platform": "pc",
    "language": "en"
}

def load_arcanes_by_rank(filename="arcanes_by_rank.txt"):
    arcane_ranks = defaultdict(dict)
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) == 3 and parts[1] == "rank":
                name = parts[0]
                rank = int(parts[2])
                arcane_ranks[name][rank] = f"{name}_r{rank}"
    return arcane_ranks

def get_orders(item_url_name):
    url = f"{BASE_URL}/items/{item_url_name}/orders"
    try:
        response = requests.get(url, headers=HEADERS)
        orders = response.json()["payload"]["orders"]
        filtered = [o for o in orders if o["order_type"] == "sell" and o["user"]["status"] == "ingame"]
        return sorted(filtered, key=lambda o: o["platinum"])
    except Exception as e:
        print(f"❌ Error fetching {item_url_name}: {e}")
        return []

def get_lowest_sell(item_url_name):
    orders = get_orders(item_url_name)
    return orders[0]["platinum"] if orders else None

def analyze_arcanes(arcanes):
    results = []

    print("\n🔮 ARCANE RANK SPREAD ANALYSIS")
    print("-" * 60)

    for arcane_name, ranks in arcanes.items():
        rank0_url = ranks.get(0)
        rank5_url = ranks.get(5)
        if not rank0_url or not rank5_url:
            print(f"{arcane_name}: Missing rank 0 or 5, skipping.")
            continue

        price_r0 = get_lowest_sell(rank0_url)
        price_r5 = get_lowest_sell(rank5_url)

        if price_r0 is None or price_r5 is None:
            print(f"{arcane_name}: Missing price data, skipping.")
            continue

        difference = price_r5 - price_r0
        percentage = (difference / price_r0) * 100 if price_r0 else 0

        print(f"🔸 {arcane_name}")
        print(f"  Rank 0: {price_r0}p | Rank 5: {price_r5}p")
        print(f"  Difference: {difference:.1f}p | % Diff: {percentage:.1f}%\n")

        results.append({
            "name": arcane_name,
            "rank_0": price_r0,
            "rank_5": price_r5,
            "difference": difference,
            "percent_diff": percentage
        })

        time.sleep(0.5)

    return results

def save_results(results, filename="arcane_spread_analysis.txt"):
    with open(filename, "w") as f:
        f.write("📊 Sorted by Absolute Platinum Difference:\n")
        for r in sorted(results, key=lambda x: x["difference"], reverse=True):
            f.write(f"{r['name']}: Rank 0 {r['rank_0']}p | Rank 5 {r['rank_5']}p | Diff {r['difference']}p | % {r['percent_diff']:.1f}%\n")

        f.write("\n📊 Sorted by Percentage Difference:\n")
        for r in sorted(results, key=lambda x: x["percent_diff"], reverse=True):
            f.write(f"{r['name']}: Rank 0 {r['rank_0']}p | Rank 5 {r['rank_5']}p | Diff {r['difference']}p | % {r['percent_diff']:.1f}%\n")

    print(f"\n📄 Saved to {filename}")

if __name__ == "__main__":
    arcane_data = load_arcanes_by_rank("arcanes_by_rank.txt")
    results = analyze_arcanes(arcane_data)
    save_results(results)



🔮 ARCANE RANK SPREAD ANALYSIS
------------------------------------------------------------

📄 Saved to arcane_spread_analysis.txt


In [5]:
import requests
import time
from collections import defaultdict

BASE_URL = "https://api.warframe.market/v1"
HEADERS = {
    "accept": "application/json",
    "platform": "pc",
    "language": "en"
}

def load_mods_by_rank(filename="mods_by_rank.txt"):
    mods = defaultdict(dict)
    try:
        with open(filename, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) >= 3 and parts[-2] == "rank":
                    name = " ".join(parts[:-2])
                    rank = int(parts[-1])
                    url_name = name.lower().replace(" ", "_").replace("'", "")
                    mods[name][rank] = url_name
        print(f"📂 Loaded {len(mods)} mods from {filename}")
        return mods
    except FileNotFoundError:
        print(f"❌ File {filename} not found")
        return {}

def get_orders(item_url_name, rank):
    url = f"{BASE_URL}/items/{item_url_name}_r{rank}/orders"
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        orders = response.json()["payload"]["orders"]
        filtered = [o for o in orders if o["order_type"] == "sell" and o["user"]["status"] == "ingame"]
        return sorted(filtered, key=lambda o: o["platinum"])
    except Exception as e:
        print(f"❌ Error fetching {item_url_name}_r{rank}: {e}")
        return []

def get_lowest_sell(item_url_name, rank):
    orders = get_orders(item_url_name, rank)
    return orders[0]["platinum"] if orders else None

def analyze_mods(mods):
    results = []
    print("\n📦 MOD RANK SPREAD ANALYSIS (≥5p)")
    print("-" * 60)

    for mod_name, ranks in mods.items():
        if 0 not in ranks:
            print(f"⚠️ {mod_name}: Missing Rank 0, skipping.")
            continue

        max_rank = max(ranks.keys())
        if max_rank not in ranks:
            print(f"⚠️ {mod_name}: Missing max rank ({max_rank}), skipping.")
            continue

        rank0_url = ranks[0]
        max_rank_url = ranks[max_rank]

        price_r0 = get_lowest_sell(rank0_url, 0)
        price_max = get_lowest_sell(max_rank_url, max_rank)

        if price_r0 is None or price_max is None:
            print(f"❌ {mod_name}: Missing price data (Rank 0: {price_r0}, Rank {max_rank}: {price_max}), skipping.\n")
            continue

        if price_r0 < 5 and price_max < 5:
            continue

        difference = price_max - price_r0
        percent_diff = (difference / price_r0) * 100 if price_r0 else 0

        print(f"🔸 {mod_name}")
        print(f"  Rank 0: {price_r0}p | Rank {max_rank}: {price_max}p")
        print(f"  Difference: {difference:.1f}p | % Diff: {percent_diff:.1f}%\n")

        results.append({
            "name": mod_name,
            "rank_0": price_r0,
            "rank_max": price_max,
            "max_rank": max_rank,
            "difference": difference,
            "percent_diff": percent_diff
        })

        time.sleep(0.4)

    return results

def save_results(results, filename="mod_rank_spread_analysis.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write("📊 Sorted by Absolute Platinum Difference:\n\n")
        for r in sorted(results, key=lambda x: x["difference"], reverse=True):
            f.write(f"{r['name']}: Rank 0 {r['rank_0']}p | Rank {r['max_rank']} {r['rank_max']}p "
                    f"| Diff {r['difference']}p | % {r['percent_diff']:.1f}%\n")

        f.write("\n📊 Sorted by Percentage Difference:\n\n")
        for r in sorted(results, key=lambda x: x["percent_diff"], reverse=True):
            f.write(f"{r['name']}: Rank 0 {r['rank_0']}p | Rank {r['max_rank']} {r['rank_max']}p "
                    f"| Diff {r['difference']}p | % {r['percent_diff']:.1f}%\n")

    print(f"\n📄 Results saved to '{filename}'")

if __name__ == "__main__":
    mod_data = load_mods_by_rank("mods_by_rank.txt")
    results = analyze_mods(mod_data)
    save_results(results)


📦 MOD RANK SPREAD ANALYSIS (≥5p)
------------------------------------------------------------

📄 Results saved to 'mod_rank_spread_analysis.txt'
